# X6S1 notebook
Set **Runtime → Change runtime type → T4 GPU** (any GPU offered is acceptable), then run the six cells in order (Shift+Enter each, or **Runtime → Run all**). Never edit a cell other than pasting the hash in cell 2, never run additional code, never use a CPU runtime.

**Cell 1 GPU.** Expected: `GPU OK: Tesla T4` (name may differ). `NO GPU`: set the runtime; if Colab cannot connect to a GPU (usage limit), stop and retry hours later or next day.

In [ ]:
# X6S1 cell 1: GPU
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
if not torch.cuda.is_available():
    raise SystemExit("NO GPU")
print("GPU OK:", torch.cuda.get_device_name(0))

**Cell 2 Drive.** Paste BUNDLE_SHA256 between the quotes. Expected: a Google sign-in prompt, `Mounted at /content/drive` and `ROOT X6S1/<bundle-id>`. `STOP: BUNDLE DIRECTORY`: check the upload and the pasted hash.

In [ ]:
# X6S1 cell 2: Drive
import glob
from google.colab import drive
drive.mount("/content/drive")
BUNDLE_SHA256 = "PASTE_BUNDLE_SHA256_HERE"
_hits = glob.glob("/content/drive/MyDrive/X6S1/*-" + BUNDLE_SHA256.strip().lower()[:12])
if len(_hits) != 1:
    raise SystemExit("STOP: BUNDLE DIRECTORY")
ROOT = _hits[0]
print("ROOT", ROOT[len("/content/drive/MyDrive/"):])

**Cell 3 Verify.** Expected: `BUNDLE OK …` with the wave, the models and the code prefix. `STOP: BUNDLE HASH MISMATCH`: check the pasted hash and that the upload finished; otherwise send the message to the coordinator.

In [ ]:
# X6S1 cell 3: verify
import hashlib, sys, tarfile
def _h(p): return hashlib.sha256(open(p, "rb").read()).hexdigest()
if _h(ROOT + "/SHA256SUMS") != BUNDLE_SHA256.strip().lower():
    raise SystemExit("STOP: BUNDLE HASH MISMATCH")
_sums = {l[66:]: l[:64] for l in open(ROOT + "/SHA256SUMS", encoding="utf-8").read().splitlines()}
if _h(ROOT + "/code.tar") != _sums.get("code.tar"):
    raise SystemExit("STOP: BUNDLE HASH MISMATCH")
tarfile.open(ROOT + "/code.tar").extractall("/content/x6s1")
sys.path[:0] = ["/content/x6s1/src", "/content/x6s1/tools/x6_stage1"]
import x6s1_cloud
BUNDLE = x6s1_cloud.verify_bundle(ROOT, BUNDLE_SHA256.strip().lower())

**Cell 4 Environment.** Expected: `ENV OK …` and `PRECHECK OK …`. Any `STOP:` line: send it verbatim to the coordinator; do not continue.

In [ ]:
# X6S1 cell 4: environment
ENV = x6s1_cloud.environment(ROOT, BUNDLE)
x6s1_cloud.precheck(ROOT, ENV)

**Cell 5 Train.** Resumes automatically; ends with `WAVE DONE`. On a disconnect or timeout: reconnect and run cells 1–5 again with the same BUNDLE_SHA256. `STOP:`: send it.

In [ ]:
# X6S1 cell 5: train
x6s1_cloud.run_wave(ROOT, ENV)

**Cell 6 Finalise.** Expected: `WAVE DONE` and `MANIFEST_SHA256 = <64 hex>`; send that line to the coordinator. If finalisation fails, reconnect and run cells 1–4 and then cell 6.

In [ ]:
# X6S1 cell 6: finalise
x6s1_cloud.finalise(ROOT)
drive.flush_and_unmount()